# Using OpenDP Synth - test

## Step 1: Install the library

It can be installed via the pip command:

In [ ]:
from rich.jupyter import print

%load_ext rich

In [ ]:
from lomas_client import Client
import numpy as np
import opendp.prelude as dp

## Step 2: Initialise the client

Once the library is installed, a Client object must be created. It is responsible for sending sending requests to the server and processing responses in the local environment. It enables a seamless interaction with the server. 

The client needs a few parameters to be created. Usually, these would be set in the environment by the system administrator and be transparent to lomas users. In this instance, the following code snippet sets a few of these parameters that are specific to this notebook. 

In [ ]:
# The following would usually be set in the environment by a system administrator
# and be tranparent to lomas users.
# Uncomment them if you are running against a Kubernetes deployment.
# They have already been set for you if you are running locally within a devenv or the Jupyter lab set up by Docker compose.

import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "https://lomas.example.com:443"
# os.environ["LOMAS_CLIENT_OIDC_DISCOVERY_URL"] = "https://dex.example.com:443/.well-known/openid-configuration"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# We set these ones because they are specific to this notebook.

os.environ["LOMAS_CLIENT_USER_NAME"] = "mr.corona@example.com"
os.environ["LOMAS_CLIENT_USER_PASSWORD"] = "mr.corona"
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "COVID_SYNTHETIC"

# Note that all client settings can also be passed as keyword arguments to the Client constructor.
# eg. client = Client(user_name = "Mr.corona") takes precedence over setting the "LOMAS_CLIENT_USER_NAME"
# environment variable.

In [ ]:
client = Client()

## Step 3: Metadata and dummy dataset

### Getting dataset metadata

The user has never seen the data and as a first step to understand what is available to her, she would like to check the metadata of the dataset. Therefore, she just needs to call the `get_dataset_metadata()` function of the client. As this is public information, this does not cost any budget.

This function returns metadata information in a format based on [SmartnoiseSQL dictionary format](https://docs.smartnoise.org/sql/metadata.html#dictionary-format), where among other, there is information about all the available columns, their type, bound values (see Smartnoise page for more details). Any metadata is required for Smartnoise-SQL is also required here and additional information such that the different categories in a string type column column can be added.

In [ ]:
covid_metadata = client.get_dataset_metadata()
covid_metadata


{
    '@context': ['http://www.w3.org/ns/csvw', '/home/onyxia/work/csvw-eo/csvw-eo-context.jsonld'],
    '@type': 'Table',
    'privacyUnit': 'id',
    'maxContributions': 52,
    'maxLength': 50048,
    'publicLength': 50048,
    'tableSchema': {
        'columns': [
            {
                '@type': 'Column',
                'name': 'id',
                'datatype': <DataTypes.POSITIVE_INTEGER: 'positiveInteger'>,
                'required': True,
                'privacyId': True,
                'nullableProportion': 0.0,
                'minimum': 1,
                'maximum': 2000
            },
            {
                '@type': 'Column',
                'name': 'date',
                'datatype': <DataTypes.DATE: 'date'>,
                'required': True,
                'privacyId': False,
                'nullableProportion': 0.0,
                'minimum': '2022-08-01',
                'maximum': '2023-07-30'
            },
            {
                '@type': 'C

### Get a dummy dataset

Now, that she has seen and understood the metadata, she wants to get an even better understanding of the dataset (but is still not able to see it). A solution to have an idea of what the dataset looks like it to create a dummy dataset. 

Based on the public metadata of the dataset, a random dataframe can be created created. By default, there will be 100 rows and the seed is set to 42 to ensure reproducibility, but these 2 variables can be changed to obtain different dummy datasets.
Getting a dummy dataset does not affect the budget as there is no differential privacy here. It is not a synthetic dataset and all that could be learn here is already present in the public metadata (it is created randomly on the fly based on the metadata).

Dr. FSO first create a dummy dataset with 200 rows and chooses a seed of 0.

In [ ]:
columns = ['country', 'subType', 'hospitalization', 'death', "temporal", "date"]
res = client.opendp_synth.query(epsilon=100.0, delta=0.001, columns=columns)
res

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 columns = ['country', 'subType', 'hospitalization', 'death', "temporal", "date"]             │
│ ❱ 2 res = client.opendp_synth.query(epsilon=100.0, delta=0.001, columns=columns)                 │
│   3 res                                                                                          │
│   4                                                                                              │
│                                                                                                  │
│ /home/bstuder/work/lomas/client/lomas_client/libraries/opendp_synth.py:165 in query              │
│                                                                                                  │
│   162 │   │   else:                                                                              │
│   163 │   │   │   endpoint = "opendp_synth_query"                                                │
│   164 │   │   │   request_model = OpenDPSynthDataQueryModel                                      │
│ ❱ 165 │   │   body = request_model.model_validate(body_json)                                     │
│   166 │   │   # breakpoint()                                                                     │
│   167 │   │   res = self.http_client.post(endpoint, body)                                        │
│   168                                                                                            │
│                                                                                                  │
│ /nix/store/xf2v226icxcqaa149mgi6ri3xks69p86-lomas-dev-env/lib/python3.14/site-packages/pydantic/ │
│ main.py:732 in model_validate                                                                    │
│                                                                                                  │
│    729 │   │   │   │   code='validate-by-alias-and-name-false',                                  │
│    730 │   │   │   )                                                                             │
│    731 │   │                                                                                     │
│ ❱  732 │   │   return cls.__pydantic_validator__.validate_python(                                │
│    733 │   │   │   obj,                                                                          │
│    734 │   │   │   strict=strict,                                                                │
│    735 │   │   │   extra=extra,                                                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ValidationError: 1 validation error for OpenDPSynthDataQueryModel
opendp_json
  Field required [type=missing, input_value={'dataset_name': 'COVID_S...ne, 'approx_zcdp': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [ ]:
res.result.value.describe()

In [ ]:
import polars as pl
df = pl.scan_csv("/home/lancelot/dsccadminch/lomas/server/data/datasets/covid_synthetic_data.csv")

In [ ]:
df.select(columns).collect().describe()

In [ ]:
# THIS will FAIL because all keys and cuts for those columns is given
# Need to use delta = 0 in that case
columns = ['country', 'subType', 'hospitalization', 'death', "temporal"]
res = client.opendp_synth.query(epsilon=100.0, delta=0.001, columns=columns)
print(res)

In [ ]:
# WORKING
columns = ['country', 'subType', 'hospitalization', 'death', "temporal"]
res = client.opendp_synth.query(epsilon=100.0, columns=columns)
print(res)

# test TITANIC

In [ ]:
import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "https://lomas.example.com:443"
# os.environ["LOMAS_CLIENT_OIDC_DISCOVERY_URL"] = "https://dex.example.com:443/.well-known/openid-configuration"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# We set these ones because they are specific to this notebook.

os.environ["LOMAS_CLIENT_USER_NAME"] = "jack@example.com"
os.environ["LOMAS_CLIENT_USER_PASSWORD"] = "jack"
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "TITANIC"
client = Client()

In [ ]:
res = client.opendp_synth.query(epsilon=5.0, delta = 0.001)
res